# Imports


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import os
from pathlib import Path
from bs4 import BeautifulSoup
from io import StringIO

# Data Collection


## Get AnAge Data


### Test With Small Single Page Data

In [ ]:
page = 1
anage_data_url = (
    f"https://genomics.senescence.info/species/query.php?show=1&sort=2&page={page}"
)

try:
    res = requests.get(anage_data_url)
    res.raise_for_status()
    html = res.text

    tables = pd.read_html(StringIO(str(html)))

except requests.exceptions.RequestException as e:
    print(e)

In [ ]:
data_df = tables[1]
data_df.drop(columns=["All", "Display entry"], inplace=True)
data_df.rename(
    columns={
        "Species or taxon": "Species",
        "Common name": "Name",
        # "Longevity": "longevity",
        # "HAGRID": "ID",
    },
    inplace=True,
)
data_df

,HAGRID,Species,Name,Longevity
0,3242,Orycteropus afer,Aardvark,29.8
1,2161,Proteles cristata,Aardwolf,20
2,3296,Chilabothrus exsul,Abaco Island boa,21.7
3,570,Ciconia abdimii,Abdim's stork,21.6
4,3159,Sciurus aberti,Abert's squirrel,Not yet established
5,1180,Melozone aberti,Abert's towhee,8.6
6,2260,Genetta abyssinica,Abyssinian genet,Not yet established
7,3091,Thallomys paedulcus,Acacia rat,Not yet established
8,1323,Empidonax virescens,Acadian flycatcher,12.1
9,1419,Melanerpes formicivorus,Acorn woodpecker,17.2


In [ ]:
species_cols_to_keep = [
    # "Species",
    # "Common name",
    "Kingdom",
    "Phylum",
    "Class",
    "Order",
    "Family",
    "Genus",
    "Maximum longevity",
    "Female sexual maturity",
    "Male sexual maturity",
    "Gestation",
    "Weaning",
    "Litter size",
    "Litters per year",
    "Inter-litter interval",
    "Weight at birth",
    "Weight at weaning",
    "Adult weight",
    "Postnatal growth rate",
    "Maximum longevity residual",
    "Typical body temperature",
    "Basal metabolic rate",
    "Body mass",
    "Metabolic rate per body mass",
]

In [ ]:
data_copy = data_df.copy()
data_copy[species_cols_to_keep] = None

for idx, row in data_copy.iterrows():
    species = row["Species"].replace(" ", "_")
    species_url = (
        f"https://genomics.senescence.info/species/entry.php?species={species}"
    )

    try:
        species_res = requests.get(species_url)
        species_res.raise_for_status()

        species_html = species_res.text

        species_soup = BeautifulSoup(species_html, "html.parser")
        dl_tags = species_soup.find_all("dl")

        # species_data_series = pd.Series(np.nan, index=species_cols_to_keep)
        # species_data = []

        species_dict = {}

        for dl_tag in dl_tags:
            dts = [dt.text.strip() for dt in dl_tag.find_all("dt")]
            dds = [dt.text.strip() for dt in dl_tag.find_all("dd")]

            species_dict.update(zip(dts, dds))

            # species_data_series = pd.concat(
            #     [species_data_series, pd.Series(species_dict)]
            # )

            if "Taxonomy" in species_dict:
                lines = species_dict["Taxonomy"].replace("\xa0", "").split("\n")
                for line in lines:
                    key, value = line.split(":", 1)
                    species_dict[key] = value

        # species_data_series = species_data_series[species_cols_to_keep]

        # data_copy.loc[idx] = species_data_series

        data_copy.loc[idx, species_dict.keys()] = species_dict.values()

    except requests.exceptions.RequestException as e:
        print(e)

    # break
display(data_copy)

,HAGRID,Species,Name,Longevity,Kingdom,Phylum,Class,Order,Family,Genus,...,Entrez,Ageing Literature,Images,Internet,Synonyms,,Incubation,Clutch size,Clutches per year,Weight at hatching
0,3242,Orycteropus afer,Aardvark,29.8,Animalia,Chordata,Mammalia (Taxon entry),Tubulidentata,Orycteropodidae,Orycteropus,...,Search all databases,Search Google Scholar or Search PubMed,Google Image search,Search Google,NaN,NaN,NaN,NaN,NaN,NaN
1,2161,Proteles cristata,Aardwolf,20,Animalia,Chordata,Mammalia (Taxon entry),Carnivora,Hyaenidae,Proteles,...,Search all databases,Search Google Scholar or Search PubMed,Google Image search,Search Google,Proteles cristatus,NaN,NaN,NaN,NaN,NaN
2,3296,Chilabothrus exsul,Abaco Island boa,21.7,Animalia,Chordata,Reptilia (Taxon entry),Squamata,Boidae,Chilabothrus,...,Search all databases,Search Google Scholar or Search PubMed,Google Image search,Search Google,Epicrates exsul,No information is available on life history. P...,NaN,NaN,NaN,NaN
3,570,Ciconia abdimii,Abdim's stork,21.6,Animalia,Chordata,Aves (Taxon entry),Ciconiiformes,Ciconiidae,Ciconia,...,Search all databases,Search Google Scholar or Search PubMed,Google Image search,Search Google,NaN,NaN,,(oviparous),,
4,3159,Sciurus aberti,Abert's squirrel,Not yet established,Animalia,Chordata,Mammalia (Taxon entry),Rodentia,Sciuridae,Sciurus,...,Search all databases,Search Google Scholar or Search PubMed,Google Image search,Search Google,NaN,NaN,NaN,NaN,NaN,NaN
5,1180,Melozone aberti,Abert's towhee,8.6,Animalia,Chordata,Aves (Taxon entry),Passeriformes,Passerellidae,Melozone,...,Search all databases,Search Google Scholar or Search PubMed,Google Image search,Search Google,"Pipilo aberti, Kieneria aberti",NaN,14 days,3 (oviparous),1.5,3.63 g
6,2260,Genetta abyssinica,Abyssinian genet,Not yet established,Animalia,Chordata,Mammalia (Taxon entry),Carnivora,Viverridae,Genetta,...,Search all databases,Search Google Scholar or Search PubMed,Google Image search,Search Google,NaN,NaN,NaN,NaN,NaN,NaN
7,3091,Thallomys paedulcus,Acacia rat,Not yet established,Animalia,Chordata,Mammalia (Taxon entry),Rodentia,Muridae,Thallomys,...,Search all databases,Search Google Scholar or Search PubMed,Google Image search,Search Google,"Mus moggi, Thamnomys ruddi, Thallomys steven...",NaN,NaN,NaN,NaN,NaN
8,1323,Empidonax virescens,Acadian flycatcher,12.1,Animalia,Chordata,Aves (Taxon entry),Passeriformes,Tyrannidae,Empidonax,...,Search all databases,Search Google Scholar or Search PubMed,Google Image search,Search Google,NaN,NaN,13 days,3 (oviparous),1,1.9 g
9,1419,Melanerpes formicivorus,Acorn woodpecker,17.2,Animalia,Chordata,Aves (Taxon entry),Piciformes,Picidae,Melanerpes,...,Search all databases,Search Google Scholar or Search PubMed,Google Image search,Search Google,NaN,NaN,14 days,4 (oviparous),2,4.7 g


### Convert to functions and cleanup code

In [ ]:
def get_species_data(data_df):
  data_copy = data_df.copy()
  data_copy[species_cols_to_keep] = None

  for idx, row in data_copy.iterrows():
      species = row["Species"].replace(" ", "_")
      species_url = (
          f"https://genomics.senescence.info/species/entry.php?species={species}"
      )

      try:
          species_res = requests.get(species_url)
          species_res.raise_for_status()

          species_html = species_res.text

          species_soup = BeautifulSoup(species_html, "html.parser")
          dl_tags = species_soup.find_all("dl")

          # species_data_series = pd.Series(np.nan, index=species_cols_to_keep)
          # species_data = []

          species_dict = {}

          for dl_tag in dl_tags:
              dts = [dt.text.strip() for dt in dl_tag.find_all("dt")]
              dds = [dt.text.strip() for dt in dl_tag.find_all("dd")]

              species_dict.update(zip(dts, dds))

              # species_data_series = pd.concat(
              #     [species_data_series, pd.Series(species_dict)]
              # )

              if "Taxonomy" in species_dict:
                  lines = species_dict["Taxonomy"].replace("\xa0", "").split("\n")
                  for line in lines:
                      key, value = line.split(":", 1)
                      species_dict[key] = value

          # species_data_series = species_data_series[species_cols_to_keep]

          # data_copy.loc[idx] = species_data_series

          data_copy.loc[idx, species_dict.keys()] = species_dict.values()

      except requests.exceptions.RequestException as e:
          print(e)

      # break
  return data_copy

In [ ]:
def get_page(show=1, page=1):
  anage_data_url = (
      f"https://genomics.senescence.info/species/query.php?show={show}&sort=2&page={page}"
  )

  try:
      res = requests.get(anage_data_url)
      res.raise_for_status()
      html = res.text

      tables = pd.read_html(StringIO(str(html)))

  except requests.exceptions.RequestException as e:
      print(e)

  data_df = tables[1]
  data_df.drop(columns=["All", "Display entry"], inplace=True)
  data_df.rename(
      columns={
          "Species or taxon": "Species",
          "Common name": "Name",
          # "Longevity": "longevity",
          # "HAGRID": "ID",
      },
      inplace=True,
  )

  data_df = get_species_data(data_df)

  return data_df

In [ ]:
get_page(show=5, page=2)

,HAGRID,Species,Name,Longevity,Kingdom,Phylum,Class,Order,Family,Genus,...,Clutches per year,Weight at hatching,Clutch or litter size,Breedings per year,Metamorphosis,,Interbirth interval,IMR,MRDT,Resting metabolic rate
0,2757,Presbytis melalophos,"Black-crested sumatran langur, or Mitred leaf ...",20,Animalia,Chordata,Mammalia (Taxon entry),Primates (Taxon entry),Cercopithecidae,Presbytis,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1368,Nycticorax nycticorax,Black-crowned night heron,21.1,Animalia,Chordata,Aves (Taxon entry),Pelecaniformes,Ardeidae,Nycticorax,...,1,24.2 g,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1071,Oenanthe hispanica,Black-eared wheatear,4.9,Animalia,Chordata,Aves (Taxon entry),Passeriformes,Muscicapidae,Oenanthe,...,,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1716,Phalacrocorax fuscescens,Black-faced cormorant,41.6,Animalia,Chordata,Aves (Taxon entry),Suliformes,Phalacrocoracidae,Phalacrocorax,...,NaN,NaN,,,NaN,NaN,NaN,NaN,NaN,NaN
4,782,Coracina novaehollandiae,Black-faced cuckooshrike,9,Animalia,Chordata,Aves (Taxon entry),Passeriformes,Campephagidae,Coracina,...,NaN,NaN,,,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,143,Andrias davidianus,Chinese giant salamander,Not yet established,Animalia,Chordata,Amphibia (Taxon entry),Caudata,Cryptobranchidae,Andrias,...,NaN,NaN,NaN,NaN,,NaN,NaN,NaN,NaN,NaN
496,1963,Naemorhedus caudatus,Chinese goral,20.3,Animalia,Chordata,Mammalia (Taxon entry),Artiodactyla,Bovidae,Naemorhedus,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
497,2653,Manis pentadactyla,Chinese pangolin,Not yet established,Animalia,Chordata,Mammalia (Taxon entry),Pholidota,Manidae,Manis,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
498,147,Hynobius leechii,Chinese salamander,7.1,Animalia,Chordata,Amphibia (Taxon entry),Caudata,Hynobiidae,Hynobius,...,NaN,NaN,NaN,NaN,NaN,No information is available on life history. P...,NaN,NaN,NaN,NaN


### Get All Data

In [ ]:
all_data_df = pd.DataFrame([])

for page in range(1, 11):
  page_df = get_page(show=5, page=page)
  all_data_df = pd.concat([all_data_df, page_df], ignore_index=True)
  # break

all_data_df

,HAGRID,Species,Name,Longevity,Kingdom,Phylum,Class,Order,Family,Genus,...,Weight at hatching,IMR,MRDT,Metamorphosis,Resting metabolic rate,Interbirth interval,Clutch or litter size,Breedings per year,GenAge,GenDR
0,3242,Orycteropus afer,Aardvark,29.8,Animalia,Chordata,Mammalia (Taxon entry),Tubulidentata,Orycteropodidae,Orycteropus,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2161,Proteles cristata,Aardwolf,20,Animalia,Chordata,Mammalia (Taxon entry),Carnivora,Hyaenidae,Proteles,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3296,Chilabothrus exsul,Abaco Island boa,21.7,Animalia,Chordata,Reptilia (Taxon entry),Squamata,Boidae,Chilabothrus,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,570,Ciconia abdimii,Abdim's stork,21.6,Animalia,Chordata,Aves (Taxon entry),Ciconiiformes,Ciconiidae,Ciconia,...,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,3159,Sciurus aberti,Abert's squirrel,Not yet established,Animalia,Chordata,Mammalia (Taxon entry),Rodentia,Sciuridae,Sciurus,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4666,597,Geopelia striata,Zebra dove,18.7,Animalia,Chordata,Aves (Taxon entry),Columbiformes,Columbidae,Geopelia,...,NaN,NaN,NaN,NaN,NaN,NaN,,,NaN,NaN
4667,1935,Cephalophus zebra,Zebra duiker,Not yet established,Animalia,Chordata,Mammalia (Taxon entry),Artiodactyla,Bovidae,Cephalophus,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4668,882,Taeniopygia guttata,Zebra finch,12,Animalia,Chordata,Aves (Taxon entry),Passeriformes,Estrildidae,Taeniopygia,...,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4669,611,Zenaida aurita,Zenaida dove,Not yet established,Animalia,Chordata,Aves (Taxon entry),Columbiformes,Columbidae,Zenaida,...,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
all_data_df.to_csv('an_age_data.csv', index=False)